In [ ]:
!rm -rf /root/Real-ESRGAN
!git clone --depth 1 --branch AutoDL https://github.com/aksjfds/Real-ESRGAN.git /root/Real-ESRGAN
!if ! command -v ffmpeg >/dev/null 2>&1; then apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ffmpeg; fi
!ffmpeg -version | head -n 1
!ffmpeg -hide_banner -encoders 2>/dev/null | grep hevc_nvenc
!pip install -q -r /root/Real-ESRGAN/requirements.txt
!mkdir -p /root/autodl-tmp/realesrgan-runtime
!cd /root/Real-ESRGAN && python -m py_compile realesrgan.py realesrgan_fast.py realesrgan_fast_entry.py realesrgan_autodl_runtime.py realesrgan_autodl_entry.py enhance/*.py
!cd /root/Real-ESRGAN && CUDA_VISIBLE_DEVICES=0 TMPDIR=/root/autodl-tmp/realesrgan-runtime python realesrgan_autodl_entry.py --help >/dev/null


In [ ]:
SOURCE_INPUT_VIDEO = "/root/autodl-fs/cm_4.mp4"
INPUT_VIDEO = "/root/autodl-tmp/cm_4.mp4"
OUTPUT_VIDEO = "/root/realesrgan_basicvsrpp.mp4"

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2
FPS = "source"

START_TIME = 3 * 60 + 20
TEST_SECONDS = 60
PROGRESS_INTERVAL = 60.0

# BasicVSR++ compressed-video enhancement runs before Real-ESRGAN.
BASICVSRPP = True
BASICVSRPP_TRACK = 1             # 1=fidelity; 2=perceptual; 3=fixed-bitrate fidelity
BASICVSRPP_MODEL_PATH = ""       # empty: download the official checkpoint
BASICVSRPP_GPU = 0               # single RTX 4090
BASICVSRPP_FP16 = True
BASICVSRPP_CLIP_LENGTH = 9        # automatically falls back to 7 on OOM
BASICVSRPP_CLIP_OVERLAP = 2
BASICVSRPP_TILE_SIZE = 512        # quality baseline; never selects smaller
BASICVSRPP_TILE_PAD = 32
BASICVSRPP_STRENGTH = 1.0
BASICVSRPP_SCENE_THRESHOLD = 0.30

# Single-GPU pinned-memory/CUDA-stream auto tuning.
AUTO_TILE = True
MAX_TILE_SIZE = 1536
AUTO_BATCH = True
MAX_BATCH_SIZE = 32
TILE_SIZE = 512                   # quality baseline/fallback
TILE_PAD = 10
TILE_VERIFY_COVERAGE = False
BATCH_SIZE = 4
GPU_IDS = "0"

COLOR_POLICY = "preserve"
HDR_POLICY = "reject"

VIDEO_CODEC = "hevc_nvenc"
OUTPUT_PIX_FMT = "yuv420p"
CRF = 18
PRESET = "medium"
CQ = 15
NVENC_PRESET = "p7"
ENCODE_GPU = 0
AUDIO_CODEC = "copy"
AUDIO_BITRATE = "192k"


In [ ]:
import os
import shlex
import shutil
import subprocess
import sys
import time

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TMPDIR"] = "/root/autodl-tmp/realesrgan-runtime"

os.makedirs(os.path.dirname(INPUT_VIDEO), exist_ok=True)
source_stat = os.stat(SOURCE_INPUT_VIDEO)
copy_required = (
    not os.path.isfile(INPUT_VIDEO)
    or os.path.getsize(INPUT_VIDEO) != source_stat.st_size
    or os.stat(INPUT_VIDEO).st_mtime_ns < source_stat.st_mtime_ns
)
if copy_required:
    print(f"[copy] {SOURCE_INPUT_VIDEO} -> {INPUT_VIDEO}", flush=True)
    shutil.copy2(SOURCE_INPUT_VIDEO, INPUT_VIDEO)
else:
    print(f"[copy] reuse local input: {INPUT_VIDEO}", flush=True)

command = [
    sys.executable, "/root/Real-ESRGAN/realesrgan_autodl_entry.py",
    "--input", INPUT_VIDEO, "--output", OUTPUT_VIDEO,
    "--model", MODEL, "--model-path", MODEL_PATH,
    "--scale", str(SCALE), "--fps", FPS,
    "--fp16", "--channels-last",
    "--basicvsrpp" if BASICVSRPP else "--no-basicvsrpp",
    "--basicvsrpp-track", str(BASICVSRPP_TRACK),
    "--basicvsrpp-model-path", BASICVSRPP_MODEL_PATH,
    "--basicvsrpp-gpu", str(BASICVSRPP_GPU),
    "--basicvsrpp-fp16" if BASICVSRPP_FP16 else "--no-basicvsrpp-fp16",
    "--basicvsrpp-clip-length", str(BASICVSRPP_CLIP_LENGTH),
    "--basicvsrpp-clip-overlap", str(BASICVSRPP_CLIP_OVERLAP),
    "--basicvsrpp-tile-size", str(BASICVSRPP_TILE_SIZE),
    "--basicvsrpp-tile-pad", str(BASICVSRPP_TILE_PAD),
    "--basicvsrpp-strength", str(BASICVSRPP_STRENGTH),
    "--basicvsrpp-scene-threshold", str(BASICVSRPP_SCENE_THRESHOLD),
    "--auto-tile" if AUTO_TILE else "--no-auto-tile",
    "--max-tile-size", str(MAX_TILE_SIZE),
    "--auto-batch" if AUTO_BATCH else "--no-auto-batch",
    "--max-batch-size", str(MAX_BATCH_SIZE),
    "--tile-size", str(TILE_SIZE), "--tile-pad", str(TILE_PAD),
    "--tile-verify-coverage" if TILE_VERIFY_COVERAGE else "--no-tile-verify-coverage",
    "--batch-size", str(BATCH_SIZE), "--gpu-ids", GPU_IDS,
    "--color-policy", COLOR_POLICY, "--hdr-policy", HDR_POLICY,
    "--video-codec", VIDEO_CODEC, "--output-pix-fmt", OUTPUT_PIX_FMT,
    "--crf", str(CRF), "--preset", PRESET, "--cq", str(CQ),
    "--nvenc-preset", NVENC_PRESET, "--encode-gpu", str(ENCODE_GPU),
    "--audio-codec", AUDIO_CODEC, "--audio-bitrate", AUDIO_BITRATE,
    "--start-time", str(START_TIME), "--test-seconds", str(TEST_SECONDS),
    "--progress-interval", str(PROGRESS_INTERVAL),
    "--ffmpeg-bin", "ffmpeg", "--ffprobe-bin", "ffprobe",
]
print("[command]", shlex.join(command), flush=True)
started = time.monotonic()
subprocess.run(command, check=True)
print(f"[notebook] elapsed={time.monotonic() - started:.1f}s", flush=True)
